# Bước 3.5: Export split và metadata - Intel Image Classification

Notebook này khóa dữ liệu thí nghiệm cho các notebook train sau bằng cách tạo split cố định và export metadata vào `data/processed/`.


## Mục tiêu notebook

- Giữ nguyên test set làm tập đánh giá.
- Tách validation từ train set bằng stratified split.
- Lưu split và metadata cố định để các notebook train sau dùng chung.
- Notebook này không huấn luyện mô hình.


In [1]:
from pathlib import Path
import os
import json
import random

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
VAL_RATIO = 0.2
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


In [2]:
def find_project_root(start_path=None):
    """Tìm project root dựa trên parent gần nhất có thư mục data."""
    current = Path.cwd() if start_path is None else Path(start_path).resolve()
    candidates = [current, *current.parents]

    for candidate in candidates:
        if (candidate / "data").exists():
            return candidate

    return current


PROJECT_ROOT = find_project_root()
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"

print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir: {DATA_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")


Project root: d:\CITD\HK3\ML\intel-image-classification
Data dir: d:\CITD\HK3\ML\intel-image-classification\data
Processed dir: d:\CITD\HK3\ML\intel-image-classification\data\processed


In [3]:
def is_image_file(path):
    return path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS


def count_direct_images(directory):
    return sum(1 for path in directory.iterdir() if is_image_file(path))


def has_class_folders(directory):
    """Kiểm tra thư mục có chứa class folders hay không."""
    if not directory.exists() or not directory.is_dir():
        return False

    child_dirs = [path for path in directory.iterdir() if path.is_dir()]
    if not child_dirs:
        return False

    return any(any(is_image_file(file_path) for file_path in child_dir.rglob("*")) for child_dir in child_dirs)


def has_prediction_images(directory):
    """Kiểm tra thư mục có ảnh pred trực tiếp hoặc dưới các tầng con."""
    return directory.exists() and directory.is_dir() and any(is_image_file(path) for path in directory.rglob("*"))


def resolve_class_root(base_dir):
    """Resolve wrapper folder kiểu seg_train/seg_train xuống tầng chứa class folders."""
    if has_class_folders(base_dir):
        return base_dir

    nested_same_name = base_dir / base_dir.name
    if has_class_folders(nested_same_name):
        return nested_same_name

    child_dirs = [path for path in base_dir.iterdir() if path.is_dir()] if base_dir.exists() else []
    if len(child_dirs) == 1 and has_class_folders(child_dirs[0]):
        return child_dirs[0]

    return base_dir


def resolve_prediction_root(base_dir):
    """Resolve wrapper folder kiểu seg_pred/seg_pred xuống tầng chứa ảnh pred."""
    nested_same_name = base_dir / base_dir.name
    if nested_same_name.exists() and has_prediction_images(nested_same_name):
        return nested_same_name

    if count_direct_images(base_dir) > 0:
        return base_dir

    child_dirs = [path for path in base_dir.iterdir() if path.is_dir()] if base_dir.exists() else []
    if len(child_dirs) == 1 and has_prediction_images(child_dirs[0]):
        return child_dirs[0]

    return base_dir


def find_dataset_paths(project_root):
    """Dò path dữ liệu theo cấu trúc ưu tiên và fallback."""
    candidates = {
        "train": [
            project_root / "data" / "seg_train",
            project_root / "data" / "intel-image-classification" / "seg_train",
        ],
        "test": [
            project_root / "data" / "seg_test",
            project_root / "data" / "intel-image-classification" / "seg_test",
        ],
        "pred": [
            project_root / "data" / "seg_pred",
            project_root / "data" / "intel-image-classification" / "seg_pred",
        ],
    }

    dataset_paths = {}
    missing = []

    for split, split_candidates in candidates.items():
        found = None
        for candidate in split_candidates:
            if not candidate.exists() or not candidate.is_dir():
                continue

            if split in {"train", "test"}:
                resolved = resolve_class_root(candidate)
                if has_class_folders(resolved):
                    found = resolved
                    break
            else:
                resolved = resolve_prediction_root(candidate)
                if has_prediction_images(resolved):
                    found = resolved
                    break

        if found is None:
            missing.append((split, split_candidates))
        else:
            dataset_paths[split] = found

    if missing:
        message = ["Không tìm thấy đầy đủ dữ liệu Intel Image Classification.", "Các path đã thử:"]
        for split, split_candidates in missing:
            message.append(f"- {split}:")
            for path in split_candidates:
                message.append(f"  + {path}")
        raise FileNotFoundError("\n".join(message))

    return dataset_paths


def to_project_relative(path, project_root=PROJECT_ROOT):
    """Lưu path relative từ project root nếu có thể."""
    path = Path(path).resolve()
    try:
        return path.relative_to(project_root.resolve()).as_posix()
    except ValueError:
        return path.as_posix()


def collect_labeled_images(split_dir):
    """Quét ảnh theo class folder, trả về image_path và class_name."""
    rows = []
    class_dirs = sorted([path for path in split_dir.iterdir() if path.is_dir()])

    for class_dir in class_dirs:
        image_files = sorted([path for path in class_dir.rglob("*") if is_image_file(path)])
        for image_path in image_files:
            rows.append(
                {
                    "image_path": to_project_relative(image_path),
                    "class_name": class_dir.name,
                }
            )

    df = pd.DataFrame(rows, columns=["image_path", "class_name"])
    if df.empty:
        raise ValueError(f"Không tìm thấy ảnh hợp lệ trong: {split_dir}")
    return df


def collect_prediction_images(pred_dir):
    """Quét ảnh pred, không gán label."""
    image_files = sorted([path for path in pred_dir.rglob("*") if is_image_file(path)])
    return pd.DataFrame(
        [{"image_path": to_project_relative(image_path), "split": "pred"} for image_path in image_files],
        columns=["image_path", "split"],
    )


def create_class_mapping(train_df):
    """Tạo mapping class ổn định theo thứ tự alphabet."""
    class_names = sorted(train_df["class_name"].unique().tolist())
    class_to_idx = {class_name: idx for idx, class_name in enumerate(class_names)}
    idx_to_class = {str(idx): class_name for class_name, idx in class_to_idx.items()}
    return class_names, class_to_idx, idx_to_class


def split_train_val(df, val_ratio=VAL_RATIO, random_state=RANDOM_STATE):
    """Tách validation từ train bằng stratified split."""
    train_df, val_df = train_test_split(
        df,
        test_size=val_ratio,
        random_state=random_state,
        stratify=df["label"],
        shuffle=True,
    )
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True)


In [4]:
DATASET_PATHS = find_dataset_paths(PROJECT_ROOT)

print("Path dữ liệu thực tế đang dùng:")
for split, path in DATASET_PATHS.items():
    print(f"- {split}: {path}")


Path dữ liệu thực tế đang dùng:
- train: d:\CITD\HK3\ML\intel-image-classification\data\seg_train
- test: d:\CITD\HK3\ML\intel-image-classification\data\seg_test
- pred: d:\CITD\HK3\ML\intel-image-classification\data\seg_pred


In [5]:
full_train_df = collect_labeled_images(DATASET_PATHS["train"])
test_df = collect_labeled_images(DATASET_PATHS["test"])

print(f"full_train_df rows: {len(full_train_df)}")
display(full_train_df.head())

print(f"test_df rows: {len(test_df)}")
display(test_df.head())


full_train_df rows: 14034


,image_path,class_name
0,data/seg_train/buildings/0.jpg,buildings
1,data/seg_train/buildings/10006.jpg,buildings
2,data/seg_train/buildings/1001.jpg,buildings
3,data/seg_train/buildings/10014.jpg,buildings
4,data/seg_train/buildings/10018.jpg,buildings


test_df rows: 3000


,image_path,class_name
0,data/seg_test/buildings/20057.jpg,buildings
1,data/seg_test/buildings/20060.jpg,buildings
2,data/seg_test/buildings/20061.jpg,buildings
3,data/seg_test/buildings/20064.jpg,buildings
4,data/seg_test/buildings/20073.jpg,buildings


In [6]:
class_names, class_to_idx, idx_to_class = create_class_mapping(full_train_df)

full_train_df["label"] = full_train_df["class_name"].map(class_to_idx).astype(int)
test_df["label"] = test_df["class_name"].map(class_to_idx)

if test_df["label"].isna().any():
    missing_classes = sorted(test_df.loc[test_df["label"].isna(), "class_name"].unique().tolist())
    raise ValueError(f"Test set có class không tồn tại trong train mapping: {missing_classes}")

test_df["label"] = test_df["label"].astype(int)

print(f"Số class: {len(class_names)}")
print("class_to_idx:")
print(json.dumps(class_to_idx, indent=2, ensure_ascii=False))
print("\nidx_to_class:")
print(json.dumps(idx_to_class, indent=2, ensure_ascii=False))


Số class: 6
class_to_idx:
{
  "buildings": 0,
  "forest": 1,
  "glacier": 2,
  "mountain": 3,
  "sea": 4,
  "street": 5
}

idx_to_class:
{
  "0": "buildings",
  "1": "forest",
  "2": "glacier",
  "3": "mountain",
  "4": "sea",
  "5": "street"
}


In [7]:
train_df, val_df = split_train_val(full_train_df, val_ratio=VAL_RATIO, random_state=RANDOM_STATE)

train_df = train_df.copy()
val_df = val_df.copy()
test_df = test_df.copy()

train_df["split"] = "train"
val_df["split"] = "val"
test_df["split"] = "test"

train_df = train_df[["image_path", "class_name", "label", "split"]]
val_df = val_df[["image_path", "class_name", "label", "split"]]
test_df = test_df[["image_path", "class_name", "label", "split"]]

print(f"Số mẫu train: {len(train_df)}")
print(f"Số mẫu val: {len(val_df)}")
print(f"Số mẫu test: {len(test_df)}")

distribution_df = (
    pd.concat([train_df, val_df, test_df], ignore_index=True)
    .groupby(["split", "class_name", "label"])
    .size()
    .reset_index(name="num_images")
    .sort_values(["split", "label"])
    .reset_index(drop=True)
)

display(distribution_df)


Số mẫu train: 11227
Số mẫu val: 2807
Số mẫu test: 3000


,split,class_name,label,num_images
0,test,buildings,0,437
1,test,forest,1,474
2,test,glacier,2,553
3,test,mountain,3,525
4,test,sea,4,510
5,test,street,5,501
6,train,buildings,0,1753
7,train,forest,1,1817
8,train,glacier,2,1923
9,train,mountain,3,2009


In [8]:
pred_df = collect_prediction_images(DATASET_PATHS["pred"])

print(f"Tổng số ảnh pred: {len(pred_df)}")
display(pred_df.head())


Tổng số ảnh pred: 7301


,image_path,split
0,data/seg_pred/10004.jpg,pred
1,data/seg_pred/10005.jpg,pred
2,data/seg_pred/10012.jpg,pred
3,data/seg_pred/10013.jpg,pred
4,data/seg_pred/10017.jpg,pred


In [9]:
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
print(f"Đã sẵn sàng thư mục export: {PROCESSED_DIR}")


Đã sẵn sàng thư mục export: d:\CITD\HK3\ML\intel-image-classification\data\processed


In [10]:
dataset_summary = {
    "num_classes": len(class_names),
    "class_names": class_names,
    "num_train": int(len(train_df)),
    "num_val": int(len(val_df)),
    "num_test": int(len(test_df)),
    "num_pred": int(len(pred_df)),
    "random_state": RANDOM_STATE,
    "val_ratio": VAL_RATIO,
}

print(json.dumps(dataset_summary, indent=2, ensure_ascii=False))


{
  "num_classes": 6,
  "class_names": [
    "buildings",
    "forest",
    "glacier",
    "mountain",
    "sea",
    "street"
  ],
  "num_train": 11227,
  "num_val": 2807,
  "num_test": 3000,
  "num_pred": 7301,
  "random_state": 42,
  "val_ratio": 0.2
}


In [11]:
export_paths = {
    "train_split": PROCESSED_DIR / "train_split.csv",
    "val_split": PROCESSED_DIR / "val_split.csv",
    "test_manifest": PROCESSED_DIR / "test_manifest.csv",
    "pred_manifest": PROCESSED_DIR / "pred_manifest.csv",
    "class_to_idx": PROCESSED_DIR / "class_to_idx.json",
    "idx_to_class": PROCESSED_DIR / "idx_to_class.json",
    "dataset_summary": PROCESSED_DIR / "dataset_summary.json",
}

train_df.to_csv(export_paths["train_split"], index=False)
val_df.to_csv(export_paths["val_split"], index=False)
test_df.to_csv(export_paths["test_manifest"], index=False)
pred_df.to_csv(export_paths["pred_manifest"], index=False)

export_paths["class_to_idx"].write_text(
    json.dumps(class_to_idx, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
export_paths["idx_to_class"].write_text(
    json.dumps(idx_to_class, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
export_paths["dataset_summary"].write_text(
    json.dumps(dataset_summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Đã export các artifact:")
for name, path in export_paths.items():
    print(f"- {name}: {path}")


Đã export các artifact:
- train_split: d:\CITD\HK3\ML\intel-image-classification\data\processed\train_split.csv
- val_split: d:\CITD\HK3\ML\intel-image-classification\data\processed\val_split.csv
- test_manifest: d:\CITD\HK3\ML\intel-image-classification\data\processed\test_manifest.csv
- pred_manifest: d:\CITD\HK3\ML\intel-image-classification\data\processed\pred_manifest.csv
- class_to_idx: d:\CITD\HK3\ML\intel-image-classification\data\processed\class_to_idx.json
- idx_to_class: d:\CITD\HK3\ML\intel-image-classification\data\processed\idx_to_class.json
- dataset_summary: d:\CITD\HK3\ML\intel-image-classification\data\processed\dataset_summary.json


In [12]:
expected_columns = {
    "train_split": ["image_path", "class_name", "label", "split"],
    "val_split": ["image_path", "class_name", "label", "split"],
    "test_manifest": ["image_path", "class_name", "label", "split"],
    "pred_manifest": ["image_path", "split"],
}

loaded_checks = {
    "train_split": pd.read_csv(export_paths["train_split"]),
    "val_split": pd.read_csv(export_paths["val_split"]),
    "test_manifest": pd.read_csv(export_paths["test_manifest"]),
    "pred_manifest": pd.read_csv(export_paths["pred_manifest"]),
}

for name, loaded_df in loaded_checks.items():
    actual_columns = loaded_df.columns.tolist()
    if actual_columns != expected_columns[name]:
        raise ValueError(f"{name} columns sai. Expected={expected_columns[name]}, actual={actual_columns}")
    print(f"{name}: OK, rows={len(loaded_df)}")


train_split: OK, rows=11227
val_split: OK, rows=2807
test_manifest: OK, rows=3000
pred_manifest: OK, rows=7301


In [13]:
print("Tổng kết export split và metadata:")
print(f"- Số class: {len(class_names)}")
print(f"- Danh sách class: {', '.join(class_names)}")
print(f"- Số mẫu train: {len(train_df)}")
print(f"- Số mẫu val: {len(val_df)}")
print(f"- Số mẫu test: {len(test_df)}")
print(f"- Số mẫu pred: {len(pred_df)}")
print("- File đã export:")
for name, path in export_paths.items():
    print(f"  + {name}: {path}")


Tổng kết export split và metadata:
- Số class: 6
- Danh sách class: buildings, forest, glacier, mountain, sea, street
- Số mẫu train: 11227
- Số mẫu val: 2807
- Số mẫu test: 3000
- Số mẫu pred: 7301
- File đã export:
  + train_split: d:\CITD\HK3\ML\intel-image-classification\data\processed\train_split.csv
  + val_split: d:\CITD\HK3\ML\intel-image-classification\data\processed\val_split.csv
  + test_manifest: d:\CITD\HK3\ML\intel-image-classification\data\processed\test_manifest.csv
  + pred_manifest: d:\CITD\HK3\ML\intel-image-classification\data\processed\pred_manifest.csv
  + class_to_idx: d:\CITD\HK3\ML\intel-image-classification\data\processed\class_to_idx.json
  + idx_to_class: d:\CITD\HK3\ML\intel-image-classification\data\processed\idx_to_class.json
  + dataset_summary: d:\CITD\HK3\ML\intel-image-classification\data\processed\dataset_summary.json


## Kết luận bước 3.5

Test set đã được giữ nguyên. Validation set đã được tách từ train set bằng stratified split. Split và metadata cố định đã được export vào `data/processed`.

Các notebook train sau sẽ dùng lại các file trong `data/processed` để đảm bảo thí nghiệm nhất quán.
